# 04 — Gold scoring + qualidade diária

Cria:
- `kabum_project.gold.notebooks_features_scored`
- `kabum_project.gold.catalog_quality_daily`

In [0]:
%run ./00_config_uc


# 00 — Config (Unity Catalog + ADLS)

Centraliza catálogo/schema, storage account e paths base.


In [0]:
spark.sql("USE CATALOG projeto_data_engineering")
spark.sql("USE SCHEMA kabum_project")

SOURCE_TABLE  = "projeto_data_engineering.kabum_project.notebooks_features_enriched"
SCORED_TABLE  = "projeto_data_engineering.kabum_project.notebooks_features_scored"
QUALITY_TABLE = "projeto_data_engineering.kabum_project.catalog_quality_daily"


In [0]:
from pyspark.sql import functions as F

df = spark.read.table(SOURCE_TABLE)

# Campos usados para medir completude (ajuste se quiser)
candidate_fields = [
    "ram_gb","storage_total_gb","screen_inches","cpu_model","cpu_series","cpu_gen",
    "gpu_model","screen_resolution_std","panel_type_std","refresh_rate_hz",
    "price","original_price","discount_pct","brand"
]
fields = [c for c in candidate_fields if c in df.columns]

if not fields:
    raise Exception("Nenhum campo de completude encontrado no dataset. Ajuste candidate_fields.")

# completeness_pct = % de campos preenchidos por linha
filled = sum(F.when(F.col(c).isNotNull(), F.lit(1)).otherwise(F.lit(0)) for c in fields)
df_scored = (
    df
    .withColumn("catalog_fields_total", F.lit(len(fields)))
    .withColumn("catalog_fields_filled", filled)
    .withColumn("catalog_completeness_pct", F.round(F.col("catalog_fields_filled")/F.col("catalog_fields_total")*100, 2))
    .withColumn("catalog_completeness_score", F.col("catalog_fields_filled"))  # score simples (0..N)
    .withColumn(
        "catalog_completeness_tier",
        F.when(F.col("catalog_completeness_pct") >= 80, F.lit("HIGH"))
         .when(F.col("catalog_completeness_pct") >= 50, F.lit("MEDIUM"))
         .otherwise(F.lit("LOW"))
    )
    .withColumn("ingestion_date", F.current_date())
)

(df_scored.write
 .format("delta")
 .mode("overwrite")
 .option("mergeSchema","true")
 .option("overwriteSchema","true")
 .saveAsTable(SCORED_TABLE)
)

print("OK ->", SCORED_TABLE, "rows:", df_scored.count())


OK -> projeto_data_engineering.kabum_project.notebooks_features_scored rows: 200


In [0]:
# Tabela de qualidade diária (por marketplace/search_term se existir)
df_scored = spark.read.table(SCORED_TABLE)

dims = []
for d in ["marketplace","search_term"]:
    if d in df_scored.columns:
        dims.append(d)

group_cols = ["ingestion_date"] + dims

df_q = (
    df_scored.groupBy(*group_cols)
      .agg(
          F.count("*").alias("products"),
          F.round(F.avg("catalog_completeness_pct"), 2).alias("avg_completeness_pct"),
          F.round(F.expr("percentile_approx(catalog_completeness_pct, 0.5)"), 2).alias("median_completeness_pct"),
          F.round(F.avg(F.when(F.col("catalog_completeness_tier")=="HIGH", 1).otherwise(0))*100, 2).alias("high_pct"),
          F.round(F.avg(F.when(F.col("catalog_completeness_tier")=="MEDIUM", 1).otherwise(0))*100, 2).alias("medium_pct"),
          F.round(F.avg(F.when(F.col("catalog_completeness_tier")=="LOW", 1).otherwise(0))*100, 2).alias("low_pct"),
      )
)

from delta.tables import DeltaTable

# Persistência idempotente: MERGE por (ingestion_date + dims)
key_cols = group_cols  # ['ingestion_date'] + dims

if spark.catalog.tableExists(QUALITY_TABLE):
    tgt = DeltaTable.forName(spark, QUALITY_TABLE)

    cond = " AND ".join([f"t.{c} <=> s.{c}" for c in key_cols])
    (tgt.alias("t")
        .merge(df_q.alias("s"), cond)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (df_q.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema","true")
        .saveAsTable(QUALITY_TABLE)
    )

print("OK ->", QUALITY_TABLE, "rows:", spark.read.table(QUALITY_TABLE).count())


OK -> projeto_data_engineering.kabum_project.catalog_quality_daily rows: 10


In [0]:
display(spark.read.table(QUALITY_TABLE).orderBy(F.col("ingestion_date").desc()).limit(50))

ingestion_date,marketplace,search_term,products,avg_completeness_pct,median_completeness_pct,high_pct,medium_pct,low_pct
2026-03-05,kabum,ultrabook,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,kabum,notebook ryzen,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,kabum,notebook i5,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,kabum,notebook,40,100.0,100.0,100.0,0.0,0.0
2026-03-05,kabum,notebook gamer,40,100.0,100.0,100.0,0.0,0.0
2026-03-04,kabum,notebook,20,100.0,100.0,100.0,0.0,0.0
2026-03-04,kabum,notebook gamer,20,100.0,100.0,100.0,0.0,0.0
2026-03-04,kabum,notebook i5,20,100.0,100.0,100.0,0.0,0.0
2026-03-04,kabum,notebook ryzen,20,100.0,100.0,100.0,0.0,0.0
2026-03-04,kabum,ultrabook,20,100.0,100.0,100.0,0.0,0.0
